### Notebook: 02_LDA_20NG.ipynb — Clean LDA on 20 Newsgroups
**Pipeline:** Clean (stopwords+contractions) → tokens → Dictionary/BoW → Gensim LDA → metrics → export topics.


In [1]:
# ------------------------------------------------------------
# Cell 0 — Environment & folders
# ------------------------------------------------------------
import sys, platform
from pathlib import Path
print("Python exe :", sys.executable)
print("Python ver :", platform.python_version())
ROOT=Path(r"C:\Users\murth\Desktop\nlpSession02\codeBase\topic_modeling"); DATA_DIR=ROOT/"data"; FIG_DIR=ROOT/"figures"
DATA_DIR.mkdir(parents=True, exist_ok=True); FIG_DIR.mkdir(parents=True, exist_ok=True)
print("Folders:", DATA_DIR, "|", FIG_DIR)


Python exe : c:\Users\murth\Desktop\nlpSession02\codeBase\.venv\Scripts\python.exe
Python ver : 3.10.11
Folders: C:\Users\murth\Desktop\nlpSession02\codeBase\topic_modeling\data | C:\Users\murth\Desktop\nlpSession02\codeBase\topic_modeling\figures


### Cell 1 — Load CSV


In [2]:
# ------------------------------------------------------------
# Cell 1 — Load CSV
# ------------------------------------------------------------
import numpy as np, pandas as pd
from pathlib import Path
csv = Path(r"C:\Users\murth\Desktop\nlpSession02\codeBase\topic_modeling\data\20newsgroups.csv")
df = pd.read_csv(csv); docs_raw = df["text"].astype(str).tolist(); targets = df["target"].to_numpy()
print("Docs:", len(docs_raw))


Docs: 6000


### Cell 2 — Clean (stopwords+contractions) → tokenize → Dictionary & BoW


In [3]:
# ------------------------------------------------------------
# Cell 2 — Clean tokens + Dictionary + BoW
# ------------------------------------------------------------
import sys
from pathlib import Path
from gensim.corpora import Dictionary

PROJECT_ROOT = Path(r"C:\Users\murth\Desktop\nlpSession02\codeBase\topic_modeling")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))
from utils_text import clean_corpus_to_tokens

tokenized = clean_corpus_to_tokens(docs_raw)
dictionary = Dictionary(tokenized)
dictionary.filter_extremes(no_below=10, no_above=0.6, keep_n=None)
dictionary.compactify()
corpus_bow = [dictionary.doc2bow(toks) for toks in tokenized]
print("Dictionary size:", len(dictionary), "| Example doc bow len:", len(corpus_bow[0]) if corpus_bow else 0)


Dictionary size: 6509 | Example doc bow len: 26


### Cell 3 — Train LDA and print topics


In [4]:
# ------------------------------------------------------------
# Cell 3 — Train LDA + print topics
# ------------------------------------------------------------
from gensim.models import LdaModel
num_topics, passes, iterations, chunksize = 20, 8, 50, 2000
lda = LdaModel(
    corpus=corpus_bow, id2word=dictionary, num_topics=num_topics,
    random_state=42, chunksize=chunksize, passes=passes, iterations=iterations,
    alpha='auto', eta='auto', eval_every=None
)
for t in range(num_topics):
    words = ", ".join([w for (w, wt) in lda.show_topic(t, topn=12)])
    print(f"Topic {t:02d}: {words}")


Topic 00: 10, 00, 14, 20, 15, 11, 12, 16, 50, 13, 25, 40
Topic 01: people, armenian, armenians, turkish, government, fbi, 000, said, gun, killed, population, batf
Topic 02: window, windows, file, problem, dos, mouse, run, does, work, running, openwindows, want
Topic 03: space, research, health, information, university, earth, 1993, national, new, conference, aids, year
Topic 04: power, radio, high, current, low, equipment, ground, radar, battery, want, need, new
Topic 05: god, people, does, believe, say, think, did, know, jesus, right, law, make
Topic 06: game, team, year, games, hockey, play, season, players, league, player, win, nhl
Topic 07: 04, 02, 03, san, april, __, won, 05, lost, 01, 1993, 06
Topic 08: israel, jews, israeli, arab, jewish, arabs, palestinian, islamic, palestine, people, palestinians, solution
Topic 09: file, output, entry, program, rules, section, info, entries, stream, return, build, null
Topic 10: image, software, windows, graphics, files, file, program, code, 

### Cell 4 — Metrics: Diversity + Coherence (c_v) → append


In [5]:
# ------------------------------------------------------------
# Cell 4 — Diversity + Coherence (c_v)
# ------------------------------------------------------------
import numpy as np, pandas as pd
from gensim.models.coherencemodel import CoherenceModel

topn=10
topic_words=[[w for (w,wt) in lda.show_topic(t, topn=topn)] for t in range(num_topics)]
diversity=len(set([w for tw in topic_words for w in tw]))/(num_topics*topn)
cm=CoherenceModel(topics=topic_words, texts=tokenized, dictionary=dictionary, coherence="c_v")
coh=cm.get_coherence()
print(f"Topic Diversity: {diversity:.3f} | Coherence c_v: {coh:.3f}")

cmp = DATA_DIR/"model_comparison.csv"
row=pd.DataFrame([{"model":"LDA","n_topics":num_topics,"topn":topn,"topic_diversity":diversity,"coherence_c_v":coh}])
row.to_csv(cmp, mode="a" if cmp.exists() else "w", header=not cmp.exists(), index=False)
print("Appended ->", cmp)


Topic Diversity: 0.830 | Coherence c_v: 0.660
Appended -> C:\Users\murth\Desktop\nlpSession02\codeBase\topic_modeling\data\model_comparison.csv


### Cell 5 — Export LDA topics (top terms + exemplar) to CSV


In [6]:
# ------------------------------------------------------------
# Cell 5 — Export LDA topics
# ------------------------------------------------------------
import numpy as np, pandas as pd

def main_topic_for_docs(corpus, model):
    best_t, best_p = [], []
    for bow in corpus:
        dist = model.get_document_topics(bow, minimum_probability=0.0)
        tid, prob = max(dist, key=lambda x:x[1])
        best_t.append(tid); best_p.append(prob)
    return np.array(best_t), np.array(best_p)

doc_topic, doc_prob = main_topic_for_docs(corpus_bow, lda)
rows=[]
for t in range(num_topics):
    terms = ", ".join([w for (w,wt) in lda.show_topic(t, topn=10)])
    idxs = np.where(doc_topic==t)[0]; n_t=len(idxs)
    if n_t>0:
        best_i = idxs[np.argmax(doc_prob[idxs])]
        snip = " ".join(tokenized[best_i]); snip=(snip[:180]+" ...") if len(snip)>180 else snip
        ex_i=int(best_i)
    else:
        ex_i, snip = -1, ""
    rows.append({"model":"LDA","topic_id":t,"n_docs":n_t,"top_terms":terms,
                 "example_doc_idx":ex_i,"example_snippet":snip})
df_lda = pd.DataFrame(rows).sort_values("n_docs", ascending=False).reset_index(drop=True)
out = DATA_DIR/"lda_topics_preview.csv"
df_lda.to_csv(out, index=False, encoding="utf-8"); print("Saved ->", out)


Saved -> C:\Users\murth\Desktop\nlpSession02\codeBase\topic_modeling\data\lda_topics_preview.csv
